In [1]:
# standard library imports 
import ast
# third-party imports
import pandas as pd 

In [2]:
morphology = {
    "person": { # 0 
        '1': 'first person',
        '2': 'second person',
        '3': 'third person',
        'x': 'uncertain person'
    },
    "number": { # 1
        's': 'singular',
        'd': 'dual',
        'p': 'plural',
        'x': 'uncertain number'
    },
    "tense": { # 2
        'p': 'present',
        'i': 'imperfect',
        'r': 'perfect',
        's': 'resultative',
        'a': 'aorist',
        'u': 'past',
        'l': 'pluperfect',
        'f': 'future',
        't': 'future perfect',
        'x': 'uncertain tense'
    },
    "mood": { # 3 
        'i': 'indicative',
        's': 'subjunctive',
        'm': 'imperative',
        'o': 'optative',
        'n': 'infinitive',
        'p': 'participle',
        'd': 'gerund',
        'g': 'gerundive',
        'u': 'supine',
        'x': 'uncertain mood',
        'y': 'finiteness unspecified',
        'e': 'indicative or subjunctive',
        'f': 'indicative or imperative',
        'h': 'subjunctive or imperative',
        't': 'finite'
    },
    "voice": { # 4
        'a': 'active',
        'm': 'middle',
        'p': 'passive',
        'e': 'middle or passive',
        'x': 'unspecified'
    },
    "gender": { # 5
        'm': 'masculine',
        'f': 'feminine',
        'n': 'neuter',
        'p': 'masculine or feminine',
        'o': 'masculine or neuter',
        'r': 'feminine or neuter',
        'q': 'masculine, feminine or neuter',
        'x': 'uncertain gender'
    },
    "case": { # 6 
        'n': 'nominative',
        'a': 'accusative',
        'o': 'oblique',
        'g': 'genitive',
        'c': 'genitive or dative',
        'e': 'accusative or dative',
        'd': 'dative',
        'b': 'ablative',
        'i': 'instrumental',
        'l': 'locative',
        'v': 'vocative',
        'x': 'uncertain case',
        'z': 'no case'
    },
    "degree": { # 7
        'p': 'positive',
        'c': 'comparative',
        's': 'superlative',
        'x': 'uncertain degree',
        'z': 'no degree'
    },
    "strength": { # 8 
        'w': 'weak',
        's': 'strong',
        't': 'weak or strong'
    },
    "inflection": { # 9 
        'n': 'non-inflecting',
        'i': 'inflecting'
    }
}

In [3]:
df = pd.read_csv("OUTPUTS/dataframe_02_6.csv", 
                 dtype={
                     "Russian Translation": "string",
                     "English Translation": "string"}
)

def drop_unnamed_columns(df):
    """
    Drop columns whose names start with 'Unnamed', typically created
    when saving/loading DataFrames with an index in CSV format.
    """
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

df = drop_unnamed_columns(df)

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_66281/163975699.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("OUTPUTS/dataframe_02_6.csv",


In [4]:
# create sub dataframe consisting of only verbs

df_verbs = df[
    # keep only verbs 
    (df['POS'] == 'V-') 
    # remove auxiliary verbs 
    # as main verbs already cointain info on its corresponding aux verbs
    & (df["Is_Compound_Aux"]==False)]
# amount of verb forms 
verb_count = len(df_verbs)
verb_count

41525

In [5]:
# file type of the columns "col" is simple string 
# to access elements, convert the columns in list "col" to an actual list 
cols = [
    "Compound_Aux_Forms",
    "Compound_Aux_IDs",
    "Compound_Aux_Lemmas",
    "Compound_Aux_Morphology"
]

for col in cols:
    df_verbs[col] = df_verbs[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_66281/3979599355.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_verbs[col] = df_verbs[col].apply(


In [6]:
# assert that all columns starting with "Compound..." contain values of type "list" 
for col in df_verbs:
    if col.startswith("Compound"):
        col_type = df_verbs[col].apply(type).value_counts()
        #print(col_type)
        assert (df_verbs[col].apply(lambda x: isinstance(x, list))).all()

In [7]:
rows = slice(0, 3)
cols = list(range(3,7)) + list(range(13, 15)) + list(range(-7, 0))

df_test = df_verbs.iloc[rows, cols]
df_test

,Sentence ID,Token ID,Form,Lemma,Morphology,Head ID,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
5,189407,2157778,дьржа,дьржати,-sppamn-si,2157784.0,дьржа землю (рѹськѹ),False,False,[],[],[],[]
11,189407,2157784,повелѣлъ,повелѣти,-sspamn-si,NaN,"азъ ѥсмь повелѣлъ ѿдати бѹицѣ (и (съ, съ, съ))",True,False,[ѥсмь],[2157785],[быти],[1spia----i]
16,189407,2157789,ѿдати,отъдати,--pna----i,2157784.0,"ѿдати бѹицѣ (и (съ, съ, съ))",False,False,[],[],[],[]


In [8]:
# make sure all Compound_Aux verbs have been removed
assert not df_verbs['Is_Compound_Aux'].any()

### CREATE QUERY

In [9]:
def match_morph(morph, pattern):
    if not isinstance(morph, str) or not isinstance(pattern, str):
        return False
    return all(
        p == '-' or (i < len(morph) and morph[i] == p)
        for i, p in enumerate(pattern)
    )

In [10]:
def aux_match(row):
    lemmas = row['Compound_Aux_Lemmas'] or []
    morphs = row['Compound_Aux_Morphology'] or []

    pairs_in_row = list(zip(lemmas, morphs))

    if pairs is None:
        return False

    return all(
        any(
            lemma == target_lemma and match_morph(morph, target_pattern)
            for lemma, morph in pairs_in_row
        )
        for target_lemma, target_pattern in pairs
    )

In [11]:
def filter_verbs_new(df, AUX, aux_pattern, main_pattern):

    def aux_match(row):
        lemmas = row['Compound_Aux_Lemmas'] or []
        morphs = row['Compound_Aux_Morphology'] or []

        pairs_in_row = list(zip(lemmas, morphs))

        # there must be a pair for each couple
        for target_lemma, target_pattern in zip(AUX, aux_pattern):
            if not any(
                lemma == target_lemma and match_morph(morph, target_pattern)
                for lemma, morph in pairs_in_row
            ):
                return False

        return True

    mask_aux = df.apply(aux_match, axis=1)

    mask_main = df['Morphology'].apply(
        lambda x: isinstance(x, str) and match_morph(x, main_pattern)
    )

    return df.loc[mask_main & mask_aux]

In [12]:
df_res = filter_verbs_new(
    df_verbs,
    AUX=['быти', 'быти'],
    aux_pattern=["-s-------", "-----------"],
    main_pattern="-sspamn-si"
)

In [13]:
df_res

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
11,mst,Mstislav’s letter,orv,189407,2157784,повелѣлъ,повелѣти,повелети,False,False,...,False,NaN,Novgorod,"азъ ѥсмь повелѣлъ ѿдати бѹицѣ (и (съ, съ, съ))",True,False,[ѥсмь],[2157785],[быти],[1spia----i]
116,mst,Mstislav’s letter,orv,189417,2157888,далъ,дати,дати,False,False,...,False,NaN,Novgorod,"ꙗ ѥсмь далъ блюдо (серебрьно, въ)",True,False,[ѥсмь],[2157889],[быти],[1spia----i]
127,mst,Mstislav’s letter,orv,189702,2157899,велѣлъ,велѣти,велети,False,False,...,False,NaN,Novgorod,ѥсмь велѣлъ бити,True,False,[ѥсмь],[2157900],[быти],[1spia----i]
287,mstislav-col,Colophon to Mstislav’s Gospel book,orv,213363,2305796,казалъ,казати,казати,False,False,...,False,NaN,Novgorod,мьстиславъ бѧшеть казалъ ѥже,True,False,[бѧшеть],[2305795],[быти],[3siia----i]
487,birchbark,109,orv,210149,2287509,кѹпилъ,купити,купити,False,False,...,False,NaN,Novgorod,еси кѹпилъ робѹ,True,False,[еси],[2287510],[быти],[2spia----i]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234077,pvl-hyp,6420,orv,202822,2249508,писалъ,пьсати,п_сати,False,False,...,False,NaN,Pskov,будеть писалъ наслѣдити имѣнье,True,False,[будеть],[2249507],[быти],[3sfia----i]
234364,pvl-hyp,6420,orv,204112,2249792,поставилъ,поставити,поставити,False,False,...,False,NaN,Pskov,бѣ поставилъ иже,True,False,[бѣ],[2249791],[быти],[3siia----i]
234374,pvl-hyp,6420,orv,202882,2249801,въпрошалъ,въпрошати,в_прошати,False,False,...,False,NaN,Pskov,бѣ въпрошалъ,True,False,[бѣ],[2249798],[быти],[3siia----i]
234471,pvl-hyp,6420,orv,202899,2249899,поставилъ,поставити,поставити,False,False,...,False,NaN,Pskov,бѣхъ поставилъ егоже,True,False,[бѣхъ],[2249898],[быти],[1siia----i]


In [14]:
df_res["Compound_Aux_Lemmas"].apply(tuple).value_counts()

Compound_Aux_Lemmas
(быти,)         569
(быти, быти)      6
Name: count, dtype: int64

In [15]:
df_filtered = df_res[
    df_res["Compound_Aux_Lemmas"].apply(tuple) == ('быти', 'быти')
]

In [16]:
df_filtered

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
125548,drac,The Tale of Dracula,orv,200267,2233879,повѣдалъ,повѣдати,поведати,False,False,...,True,не,NorthernRussia,бы еси не повѣдалъ ȥлато,True,False,"[бы, еси]","[2233875, 2233877]","[быти, быти]","[3saia----i, 2spia----i]"
125999,drac,The Tale of Dracula,orv,200543,2234324,ѿвѣщал,отъвѣщати,от_вещати,False,False,...,True,не,NorthernRussia,бы еси не ѿвѣщал,True,False,"[бы, еси]","[2234319, 2234321]","[быти, быти]","[2saia----i, 2spia----i]"
126003,drac,The Tale of Dracula,orv,200543,2234327,был,быти,быти,False,False,...,False,NaN,NorthernRussia,бы еси был на,True,False,"[бы, еси]","[2234326, 2234328]","[быти, быти]","[2saia----i, 2spia----i]"
170898,domo,45,orv,162621,2031248,былъ,быти,быти,False,False,...,False,NaN,Moscow,"дворъ бы бы былъ или гороженъ, тыненъ",True,False,"[бы, бы]","[2031247, 2031253]","[быти, быти]","[3saia----i, 3saia----i]"
173470,domo,52,orv,163764,2038234,былъ,быти,быти,False,False,...,False,NaN,Moscow,"запасъ (всѧкои) жита (всѧкое, не згноено) и бы...",True,False,"[бы, бы]","[2038231, 2038235]","[быти, быти]","[3saia----i, 3saia----i]"
227498,suz-lav,6685,orv,212117,2295936,шелъ,ити,ити,False,False,...,False,NaN,Moscow,ѥси былъ шелъ,True,False,"[ѥси, былъ]","[2295937, 2295938]","[быти, быти]","[2spia----i, -sspamn-si]"
